# Clouddex — train the cloud classifier on Google Colab (free GPU)

Trains the MobileNetV2 cloud-genus classifier and exports a **TensorFlow.js
Layers model** for the Clouddex PWA — all in the cloud, no local install, and
**no mid-notebook restart**.

**How to use**
1. Runtime → **Change runtime type** → Hardware accelerator = **GPU** → Save.
2. Run the cells top to bottom.
3. Provide the **CCSN** dataset when prompted (upload a `.zip`, mount Drive, or URL).
4. The last cell downloads `clouddex_model.zip`. Unzip it into the repo's
   `public/model/` folder, commit, and push — the live site redeploys with the
   real model.

**Why Keras 2 (`tf_keras`)?** The TF.js converter can't freeze Keras 3
SavedModels (`model.export()` → "Identity is not in graph"). Training with
`tf_keras` and exporting a Layers model avoids that entirely.

**Preprocessing contract:** images are normalized to **[-1, 1]** in the data
pipeline, and the model takes [-1,1] input directly (no preprocessing layer
inside). This matches `src/ml/preprocess.ts` in the app. Keep them in sync.

In [ ]:
# 1. Install Keras 2 + the TF.js converter BEFORE importing TF, then check GPU
#    (installing first avoids breaking an already-loaded TensorFlow.)
import os
os.system("pip install -q tf_keras tensorflowjs")
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras
print("TensorFlow", tf.__version__, "| tf_keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE — set Runtime > Change runtime type > GPU")

## 2. Get the CCSN dataset

Pick **one** option. Each class must end up as its own folder under `DATA_DIR`
(CCSN's 2-letter codes `Ci/ Cu/ …` or full names — the next cell normalizes them).

In [ ]:
# --- Option A (default): upload a .zip of the dataset ---
import os, zipfile
from google.colab import files

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
uploaded = files.upload()  # choose your CCSN .zip
for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(DATA_DIR)
        print("extracted", name)
print("top-level:", sorted(os.listdir(DATA_DIR)))

In [ ]:
# --- Option B: mount Google Drive ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/CCSN"

# --- Option C: download a .zip from a URL ---
# import os, zipfile, urllib.request
# DATA_DIR = "/content/data"; os.makedirs(DATA_DIR, exist_ok=True)
# urllib.request.urlretrieve("https://example.com/CCSN.zip", "/content/ccsn.zip")
# with zipfile.ZipFile("/content/ccsn.zip") as z: z.extractall(DATA_DIR)

## 3. Prepare the dataset (structure + cleaning)

Handles messy real-world zips, especially **Mac-made zips** (which add a
`__MACOSX` folder of `._` AppleDouble files and often nest the images one level
down). Deletes the junk, descends to the real class-folder root, renames CCSN's
2-letter codes, and drops `._`/undecodable files.

In [ ]:
# 3. Fix structure, rename classes, clean undecodable files
import os, shutil, tensorflow as tf
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False

shutil.rmtree(os.path.join(DATA_DIR, "__MACOSX"), ignore_errors=True)

def _has_images(d):
    try:
        return any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in os.listdir(d))
    except Exception:
        return False

def _find_class_root(base):
    cur = base
    for _ in range(6):
        subs = [d for d in os.listdir(cur)
                if os.path.isdir(os.path.join(cur, d)) and not d.startswith("__")]
        if len(subs) >= 2 and any(_has_images(os.path.join(cur, s)) for s in subs):
            return cur
        if len(subs) == 1:
            cur = os.path.join(cur, subs[0]); continue
        return cur
    return cur

DATA_DIR = _find_class_root(DATA_DIR)
print("DATA_DIR =", DATA_DIR)

MAP = {"Ci": "cirrus", "Cs": "cirrostratus", "Cc": "cirrocumulus",
       "Ac": "altocumulus", "As": "altostratus", "Cu": "cumulus",
       "Cb": "cumulonimbus", "Ns": "nimbostratus", "Sc": "stratocumulus",
       "St": "stratus", "Ct": "contrail"}
for code_, full in MAP.items():
    s, d = os.path.join(DATA_DIR, code_), os.path.join(DATA_DIR, full)
    if os.path.isdir(s) and not os.path.isdir(d):
        os.rename(s, d)

classes = sorted(d for d in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith("__"))
removed = 0
for c in classes:
    cdir = os.path.join(DATA_DIR, c)
    for fn in list(os.listdir(cdir)):
        fp = os.path.join(cdir, fn)
        if fn.startswith("._") or not os.path.isfile(fp):
            if os.path.isfile(fp):
                os.remove(fp); removed += 1
            continue
        try:
            tf.image.decode_image(tf.io.read_file(fp), expand_animations=False)
        except Exception:
            try:
                Image.open(fp).convert("RGB").save(fp, "JPEG", quality=92)
            except Exception:
                os.remove(fp); removed += 1

print("classes:", classes)
print("counts:", {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes})
print("removed junk/undecodable:", removed)

## 4. Build datasets (normalized to [-1, 1])

Normalization and light augmentation happen here, in the data pipeline — so the
model itself takes [-1,1] input with no preprocessing layers (which keeps it
clean to convert and matches `src/ml/preprocess.ts`).

In [ ]:
# 4. train/val datasets + class weights (CCSN is imbalanced)
import os, tensorflow as tf, tf_keras as keras
IMG_SIZE, BATCH = 224, 32

train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="categorical")
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=1337,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="categorical")

class_names = train_ds.class_names
print("MODEL OUTPUT ORDER (this becomes labels.json):", class_names)

counts = [len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names]
total, n = sum(counts), len(class_names)
class_weight = {i: (total / (n * c) if c else 1.0) for i, c in enumerate(counts)}

AUTOTUNE = tf.data.AUTOTUNE
def prep_train(x, y):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.1 * 255)
    x = tf.image.random_contrast(x, 0.9, 1.1)
    x = tf.clip_by_value(x, 0.0, 255.0)
    return x / 127.5 - 1.0, y          # -> [-1, 1], matches src/ml/preprocess.ts
def prep_val(x, y):
    return x / 127.5 - 1.0, y
train_ds = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(prep_val, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
# 5. Clean MobileNetV2 transfer-learning model (input already [-1, 1])
import tf_keras as keras
from tf_keras import layers, models
from tf_keras.applications import MobileNetV2

base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                   weights="imagenet")
base.trainable = False

model = models.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# 6. Train the head (feature extraction)
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight)

In [ ]:
# 7. Fine-tune the top of the backbone at a low LR
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight)

## 8. Export to TensorFlow.js + download

Exports a **Layers model** straight from the Keras model (no SavedModel
freezing), writes `labels.json` in the model's output order, zips, and downloads.

In [ ]:
# 8. Export TF.js Layers model -> package -> download
import os, json, shutil, tensorflowjs as tfjs

shutil.rmtree("/content/web_model", ignore_errors=True)
try:
    tfjs.converters.save_keras_model(model, "/content/web_model")
except Exception as e:
    print("save_keras_model failed -> H5 + CLI fallback:", e)
    model.save("/content/model.h5")
    import subprocess
    r = subprocess.run(
        ["tensorflowjs_converter", "--input_format=keras",
         "/content/model.h5", "/content/web_model"],
        capture_output=True, text=True)
    print(r.stdout[-1000:]); print(r.stderr[-2000:])
    assert r.returncode == 0, "conversion failed — paste the output above"

json.dump({"labels": class_names}, open("/content/web_model/labels.json", "w"), indent=2)
print("web_model:", os.listdir("/content/web_model"))

shutil.make_archive("/content/clouddex_model", "zip", "/content/web_model")
from google.colab import files
files.download("/content/clouddex_model.zip")

## 9. Install the model in the app

1. Unzip `clouddex_model.zip` — you'll get `model.json`, `group1-shard*.bin`
   weight shards, and `labels.json`.
2. Put all of them into the repo's **`public/model/`** folder (replacing the
   placeholder `labels.json`).
3. Commit and push:
   ```bash
   git add public/model && git commit -m "Add trained model" && git push
   ```
4. The GitHub Actions deploy runs automatically; the live site loses its
   **"demo model"** badge and starts making real predictions.

The app loads this Layers model with `tf.loadLayersModel` and feeds it [-1,1]
images (`src/ml/preprocess.ts`) — exactly what it was trained on.